## Import packages

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy import integrate
import os

import ADFWI
from ADFWI.model import AnisotropicElasticModel
from ADFWI.propagator import ElasticPropagator, GradProcessor
from ADFWI.survey import Receiver, SeismicData, Source, Survey
from ADFWI.utils import wavelet
from ADFWI.utils.first_arrivel_picking import mask
from ADFWI.view import plot_bcx_bcz, plot_damp
from ADFWI.fwi import ElasticFWI
from ADFWI.fwi.misfit import Misfit_waveform_L2

project_path = "./data"
os.makedirs(os.path.join(project_path,"model"), exist_ok=True)
os.makedirs(os.path.join(project_path,"waveform"), exist_ok=True)
os.makedirs(os.path.join(project_path,"survey"), exist_ok=True)
os.makedirs(os.path.join(project_path,"inversion"), exist_ok=True)


## Define the basic model parameter

In [ ]:
device = "npu:0"         # Specify the CPU/GPU/NPU device
dtype = torch.float32     # Set data type to 32-bit floating point
backend = ADFWI.set_backend(device, dtype=dtype)
ox, oz = 0, 0             # Origin coordinates for x and z directions
nz, nx = 80, 180          # Grid dimensions in z and x directions
dx, dz = 10, 10           # Grid spacing in x and z directions
nt, dt = 1000,0.001       # Time steps and time interval
nabc = 50                 # Thickness of the absorbing boundary layer
f0 = 30                   # Initial frequency in Hz
free_surface = True       # Enable free surface boundary condition

## Define the initial velocity model

In [ ]:
# Anomaly Model
vp_true      = np.ones((nz,nx))*3000
vs_true      = np.ones((nz,nx))*1500
rho_true     = np.ones((nz,nx))*2450
epsilon_true = np.ones((nz,nx))*0.1
gamma_true   = np.ones((nz,nx))*0
delta_true   = np.ones((nz,nx))*0.05

# anomaly 0 
center_x = nx//2
center_z = nz//2
center_r = 10
mask = vp_true == -1
for i in range(nz):
    for j in range(nx):
        if np.sqrt((i-center_z)**2 + (j-center_x)**2)<center_r:
            mask[i,j] = True
epsilon_true[mask] = 0.15
delta_true[mask] = 0.1
# anomaly 1 
center_x = nx//4
center_z = nz//2
mask = vp_true == -1
length_square = 20
mask[center_z-length_square//2:center_z+length_square//2,center_x-length_square//2:center_x+length_square//2] = True
epsilon_true[mask] = 0.28
delta_true[mask] = 0.25

# anomaly 2
x1, z1 = nx // 4 * 3 - 10, nz // 2 -8
length_triangle = 30
mask = vp_true == -1
height_triangle = length_triangle / np.sqrt(3)
x2, z2 = x1 + length_triangle, z1
x3, z3 = x1 + length_triangle / 2, z1 + height_triangle
for i in range(z1, int(z1 + height_triangle)):
    width = length_triangle * (1 - abs(i - z3) / height_triangle)
    start_x = int(x3 - width / 2)
    end_x = int(x3 + width / 2)
    mask[i, start_x:end_x] = True
epsilon_true[mask] = 0.2
delta_true[mask] = 0.15

# init model
vp_init      = np.ones((nz,nx))*3000
vs_init      = np.ones((nz,nx))*1500
rho_init     = np.ones((nz,nx))*2450
epsilon_init = np.ones((nz,nx))*0.1
gamma_init   = np.ones((nz,nx))*0
delta_init   = np.ones((nz,nx))*0.05

model = AnisotropicElasticModel(
                    ox,oz,nx,nz,dx,dz,
                    vp=vp_init,vs=vs_init,rho=rho_init,
                    eps=epsilon_init,gamma=gamma_init,delta=delta_init,
                    vp_grad=False,vs_grad=False,rho_grad=False,
                    eps_grad=True,gamma_grad=False,delta_grad=True,
                    eps_bound=[epsilon_true.min(),epsilon_true.max()],
                    delta_bound=[delta_true.min(),delta_true.max()],
                    free_surface=free_surface,
                    anisotropic_type='vti',
                    abc_type="PML",abc_jerjan_alpha=0.007,
                    auto_update_rho=False,
                    auto_update_vp =False,
                    nabc=nabc)
model.save(os.path.join(project_path,"model/init_model.npz"))
print(model.__repr__())

In [ ]:
# Plot the primary wave velocity (vp), vs and density (rho) of the model
model._plot_vp_vs_rho(figsize=(12,5),wspace=0.2,cbar_pad_fraction=0.18,cbar_height=0.04,cmap='coolwarm',save_path=os.path.join(project_path,"model/init_vs_vp_rho.png"))

In [ ]:
model._plot_eps_delta_gamma(figsize=(12,5),wspace=0.3,cbar_pad_fraction=-0.1,cbar_height=0.04,cmap='coolwarm',save_path=os.path.join(project_path,"model/init_epsilon_gamma_delta.png"))

## Define the observed systems: Survey = Source + Receiver

In [ ]:
# Define source positions in the model
src_z = np.array([70 for i in range(1,nx-1,5)]) 
src_x = np.array([i  for i in range(1,nx-1,5)])

# Generate wavelet for the source
src_t, src_v = wavelet(nt, dt, f0, amp0=1)  # Create time and wavelet amplitude
src_v = integrate.cumtrapz(src_v, axis=-1, initial=0)  # Integrate wavelet to get velocity

source = Source(nt=nt, dt=dt, f0=f0)  # Initialize source object

# Method 1: Add multiple sources at once (commented out)
# source.add_sources(src_x=src_x, src_z=src_z, src_wavelet=src_v, src_type='mt', src_mt=np.array([[1,0,0],[0,1,0],[0,0,1]]))

# Method 2: Loop through each source position to add them individually
for i in range(len(src_x)):
    source.add_source(src_x=src_x[i],src_z=src_z[i],src_wavelet=src_v,src_type="mt",src_mt=np.array([[1,0,0],[0,1,0],[0,0,1]]))

In [ ]:
# Define receiver positions in the model
rcv_z = np.array([10 for i in range(0,nx,1)])
rcv_x = np.array([j  for j in range(0,nx,1)])

receiver = Receiver(nt=nt, dt=dt)  # Initialize receiver object

# Method 1: Add all receivers at once (commented out)
# receiver.add_receivers(rcv_x=rcv_x, rcv_z=rcv_z, rcv_type='pr')

# Method 2: Loop through each receiver position to add them individually
for i in range(len(rcv_x)):
    receiver.add_receiver(rcv_x=rcv_x[i], rcv_z=rcv_z[i], rcv_type="pr")

In [ ]:
# survey
survey = Survey(source=source,receiver=receiver)
print(survey.__repr__())
survey.plot(model.vp,cmap='coolwarm',save_path=os.path.join(project_path,"survey/observed_system_init.png"),show=True)


In [ ]:
# Plot the wavelet used in the source
source.plot_wavelet(save_path=os.path.join(project_path,"survey/wavelets_init.png"),show=True)

## Define the propagator

In [ ]:
# Initialize the wave propagator using the specified model and survey configuration
F = ElasticPropagator(model,survey)

In [ ]:
# Retrieve the damping array from the propagator and plot it to visualize boundary conditions
if model.abc_type == "PML":
    bcx = F.bcx
    bcz = F.bcz
    title_param = {'family':'Times New Roman','weight':'normal','size': 15}
    plot_bcx_bcz(bcx,bcz,dx=dx,dz=dz,wspace=0.25,title_param=title_param,cbar_height=0.04,cbar_pad_fraction=-0.05,save_path=os.path.join(project_path,"model/boundary_condition_init.png"),show=True)
else:
    damp = F.damp
    plot_damp(damp)

## Load observed datasets

In [ ]:
# load data
d_obs = SeismicData(survey)
d_obs.load(os.path.join(project_path,"waveform/obs_data.npz"))
print(d_obs.__repr__())

## Inversion

In [ ]:
iteration = 10
# optimizer
optimizer   =   torch.optim.AdamW(model.parameters(), lr = 0.01,betas=(0.9,0.999), weight_decay=1e-4)
scheduler   =   torch.optim.lr_scheduler.StepLR(optimizer,step_size=200,gamma=0.75,last_epoch=-1)

# Setup misfit function
loss_fn = Misfit_waveform_L2(dt=dt)

# gradient processor
gradient_processor = GradProcessor()

fwi = ElasticFWI(propagator=F,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                loss_fn=loss_fn,
                obs_data=d_obs,gradient_processor=gradient_processor,
                waveform_normalize=True,
                cache_result=True,cache_gradient=True,
                save_fig_epoch=1,
                save_fig_path=os.path.join(project_path,"inversion"),
                inversion_component=["vx","vz"]
                )

fwi.forward(iteration=iteration,fd_order=4,
            batch_size=None,checkpoint_segments=1,start_iter=0)

iter_vp     = fwi.iter_vp
iter_vs     = fwi.iter_vs
iter_rho    = fwi.iter_rho
iter_eps    = fwi.iter_eps
iter_delta  = fwi.iter_delta
iter_loss   = fwi.iter_loss
np.savez(os.path.join(project_path,"inversion/iter_vp.npz"),data=np.array(iter_vp))
np.savez(os.path.join(project_path,"inversion/iter_vs.npz"),data=np.array(iter_vs))
np.savez(os.path.join(project_path,"inversion/iter_rho.npz"),data=np.array(iter_rho))
np.savez(os.path.join(project_path,"inversion/iter_eps.npz"),data=np.array(iter_eps))
np.savez(os.path.join(project_path,"inversion/iter_delta.npz"),data=np.array(iter_delta))
np.savez(os.path.join(project_path,"inversion/iter_loss.npz"),data=np.array(iter_loss))

## visualize the inverted results

In [ ]:
# plot the misfit
plt.figure(figsize=(8,6))
plt.plot(iter_loss,c='k')
plt.xlabel("Iterations", fontsize=12)
plt.ylabel("L2-norm Misfits", fontsize=12)
plt.tick_params(labelsize=12)
plt.savefig(os.path.join(project_path,"inversion/misfit.png"),bbox_inches='tight',dpi=100)
plt.show()

In [ ]:
# plot the initial model and inverted resutls
plt.figure(figsize=(12,8))
plt.subplot(121)
plt.imshow(epsilon_init,cmap='jet_r')
plt.subplot(122)
plt.imshow(iter_eps[-1],cmap='jet_r')
plt.savefig(os.path.join(project_path,"inversion/inverted_res.png"),bbox_inches='tight',dpi=100)
plt.show()